# Nvidia NemoRetriever 多文档检索示例

本notebook演示如何使用FAISS索引进行多文档检索，支持：
1. 跨文档跨页面检索（Multi-page Multi-document）
2. 每文档单页检索（保证文档多样性）

参考 M3DocRAG 的设计理念

## 1. 环境准备

In [1]:
%load_ext autoreload
%autoreload 2

# 加载 autoreload 扩展
# 设置自动重新加载模式。2 表示任何已导入的模块，只要其源代码发生变化，就会在执行下一行代码时自动重新加载

In [2]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
from transformers import AutoModel
from pdf2image import convert_from_path
from nvidia_rag_with_faiss import (
    NvidiaRAGPipeline, 
    GPUMemoryMonitor,
)

# 设置GPU设备
DEVICE = 9
torch.cuda.set_device(DEVICE)

print("✓ 环境准备完成")

/home/zechuan/miniconda3/envs/finance_RAG/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zechuan/miniconda3/envs/finance_RAG/lib/python3.11/site-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


✓ 环境准备完成


## 2. 加载Retriever模型

In [3]:
memory_monitor = GPUMemoryMonitor(DEVICE)

memory_monitor.print_memory("Before loading model")

# 加载 Nvidia NemoRetriever 模型
retriever_model = AutoModel.from_pretrained(
    'nvidia/llama-nemoretriever-colembed-3b-v1',
    device_map=f'cuda:{DEVICE}',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    revision='50c36f4d5271c6851aa08bd26d69f6e7ca8b870c',
).eval()

memory_monitor.print_memory("After loading model")
print("✓ Retriever模型加载完成")

[Before loading model] GPU Memory - Allocated: 0.00GB, Reserved: 0.00GB


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.21s/it]

[After loading model] GPU Memory - Allocated: 8.21GB, Reserved: 8.34GB
✓ Retriever模型加载完成


## 3. 加载多个文档

这里我们加载多个PDF文档，每个文档代表不同的报告

In [4]:
# 定义要加载的文档
documents = {
    "tencent_esg": "contents/2024_Tencent_ESG.pdf",
    "archi_esg": "contents/2024_architecture_ESG.pdf",
    "sanqi_esg": "contents/2024_sanqi_ESG.pdf",
    "zhongxing_esg": "contents/2024_zhongxing_ESG.pdf"
}

# 加载所有文档的图片
docid2images = {}
total_pages = 0
for doc_id, pdf_path in documents.items():
    print(f"\n加载文档: {doc_id}")
    images = convert_from_path(pdf_path, dpi=200)
    docid2images[doc_id] = images
    total_pages += len(images)
    print(f"  ✓ {doc_id}: {len(images)} 页")

print(f"\n✓ 总共加载 {len(docid2images)} 个文档")
print(f"✓ 总共 {total_pages} 页")


加载文档: tencent_esg
  ✓ tencent_esg: 111 页

加载文档: archi_esg
  ✓ archi_esg: 50 页

加载文档: sanqi_esg
  ✓ sanqi_esg: 85 页

加载文档: zhongxing_esg
  ✓ zhongxing_esg: 143 页

✓ 总共加载 4 个文档
✓ 总共 389 页


## 4. 初始化RAG Pipeline

In [5]:
import numpy as np
# 初始化 RAG Pipeline
rag_pipeline = NvidiaRAGPipeline(
    retriever_model=retriever_model,
    use_faiss=True,
    faiss_config={
        "embedding_dim": 3072,  # Nvidia NemoRetriever的嵌入维度
        "index_type": "ivfflat",  # 使用IVF索引，适合中等规模
        "nlist": int(np.sqrt(total_pages)),  # 聚类中心数量（建议为sqrt(n_pages)）
        "use_gpu": False,  # FAISS GPU索引（可选）
    },
    device=DEVICE
)

print("✓ RAG Pipeline初始化完成")

✓ RAG Pipeline初始化完成


## 5. 编码多个文档

In [6]:
# 编码所有文档
docid2embeddings = rag_pipeline.encode_documents(
    docid2images=docid2images,
    batch_size=8
)

print("\n文档编码统计:")
for doc_id, embeddings in docid2embeddings.items():
    print(f"  - {doc_id}: {embeddings.shape}")


多文档编码模式：共 4 个文档

处理文档: tencent_esg (111 页)

编码 111 个文档页面...
[Before passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 8.34GB


Extracting document embeddings...: 100%|██████████| 14/14 [01:13<00:00,  5.25s/it]


[After passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 18.12GB
✓ 文档编码完成
✓ 文档 tencent_esg 编码完成

处理文档: archi_esg (50 页)

编码 50 个文档页面...
[Before passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 8.34GB


Extracting document embeddings...: 100%|██████████| 7/7 [00:33<00:00,  4.75s/it]


[After passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 15.75GB
✓ 文档编码完成
✓ 文档 archi_esg 编码完成

处理文档: sanqi_esg (85 页)

编码 85 个文档页面...
[Before passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 8.34GB


Extracting document embeddings...: 100%|██████████| 11/11 [00:54<00:00,  4.94s/it]


[After passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 17.14GB
✓ 文档编码完成
✓ 文档 sanqi_esg 编码完成

处理文档: zhongxing_esg (143 页)

编码 143 个文档页面...
[Before passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 8.34GB


Extracting document embeddings...: 100%|██████████| 18/18 [01:27<00:00,  4.86s/it]


[After passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 19.44GB
✓ 文档编码完成
✓ 文档 zhongxing_esg 编码完成

✓ 所有文档编码完成


文档编码统计:
  - tencent_esg: torch.Size([111, 1802, 3072])
  - archi_esg: torch.Size([50, 1802, 3072])
  - sanqi_esg: torch.Size([85, 1802, 3072])
  - zhongxing_esg: torch.Size([143, 1802, 3072])


## 6. 构建统一的多文档索引
#### 使用flat索引速度太慢，四个文档要40分钟左右
#### 使用ivfflat索引(use_gpu)的话，一共只需要12分钟左右


In [7]:
# 构建统一索引
# 由于我设置use_gpu=False，所以索引保存到本地

rag_pipeline.build_unified_index(save_dir="./faiss_index/multi_doc")

print("✓ 多文档索引构建完成")


构建Token级多文档索引（M3DocRAG风格）
  - archi_esg: 50 页
  - sanqi_esg: 85 页
  - tencent_esg: 111 页
  - zhongxing_esg: 143 页

✓ Token扁平化完成：
  - 总页面数: 389
  - 总Token数: 700978
  - 嵌入形状: torch.Size([700978, 3072])

构建FAISS索引...
  训练索引...
  添加向量...
✓ 索引构建完成
  - 索引已保存: faiss_index/multi_doc/index.bin
  - 映射已保存: faiss_index/multi_doc/token2pageuid.pkl
  - Token嵌入已保存: faiss_index/multi_doc/all_token_embeddings.npy

✓ 多文档索引构建完成


In [ ]:
# 构建统一索引
# 如果设置use_gpu=True的话，保存到本地会报错
# RuntimeError: C++ exception cannot create std::vector larger than max_size()
# 这是因为 GPU 索引转换为 CPU 索引时需要分配大量连续内存，超过了系统限制。
rag_pipeline.build_unified_index(save_dir="./faiss_index/multi_doc")

print("✓ 多文档索引构建完成")


构建Token级多文档索引（M3DocRAG风格）
  - archi_esg: 50 页
  - sanqi_esg: 85 页
  - tencent_esg: 111 页
  - zhongxing_esg: 143 页

✓ Token扁平化完成：
  - 总页面数: 389
  - 总Token数: 700978
  - 嵌入形状: torch.Size([700978, 3072])


## 6. 构建统一的多文档索引

### 性能对比
- **Flat索引**: 4个文档（389页）约需 40 分钟
- **IVFFlat索引**: 4个文档（389页）约需 12 分钟

### ⚠️ 重要提示：GPU索引保存问题
对于大规模索引（700K+ 向量），GPU索引转换为CPU时可能导致内存溢出错误：
```
RuntimeError: C++ exception cannot create std::vector larger than max_size()
```

**解决方案**：
1. **推荐**: 使用 `use_gpu=False` 创建CPU索引（检索速度略慢，但可以保存）
2. 或者：不保存索引，每次重新构建（适合开发测试）
3. 或者：使用GPU索引但不保存（检索速度最快,仅在内存中使用）

In [13]:
# 构建统一索引（不保存，仅在内存中使用）
# 如果使用了 use_gpu=True，建议不保存索引以避免内存溢出
rag_pipeline.build_unified_index(save_dir=None)  # save_dir=None 表示不保存

print("✓ 多文档索引构建完成（仅在内存中）")


构建统一多文档索引
  - archi_esg: 50 页
  - sanqi_esg: 85 页
  - tencent_esg: 111 页
  - zhongxing_esg: 143 页

✓ 合并完成：总共 389 页

构建FAISS索引 (类型: ivfflat)
展平嵌入向量...


Processing pages: 100%|██████████| 389/389 [00:00<00:00, 28427.78it/s]


✓ 总共 700978 个token嵌入
✓ FAISS索引已移至GPU 9
训练索引 (nlist=19)...
✓ 索引训练完成
添加向量到索引...


Adding vectors: 100%|██████████| 87623/87623 [09:05<00:00, 160.70it/s]


✓ 索引构建完成！总共 700978 个向量


✓ 多文档索引构建完成（仅在内存中）


## 7．多文档检索示例
### 7.1跨文档跨页面检索（默认模式）
这种模式下，可以从同一文档返回多页，适合答案分散在多页的场景

In [10]:
# 定义查询
queries = [
    '针对GRI403-3，各个公司有什么区别？',
]

# 跨文档跨页面检索
results = rag_pipeline.retrieve_multi_doc(
    queries=queries,
    top_k=20,
    single_page_per_doc=False,  # 允许同一文档返回多页
)

# 显示结果
print("\n" + "="*80)
print("跨文档跨页面检索结果")
print("="*80)

for i, (query, query_results) in enumerate(zip(queries, results)):
    print(f"\n查询 {i+1}: {query}")
    print("-" * 80)
    print(query_results)
    for result in query_results[:20]:  # 显示前5个结果
        doc_id = result.get('doc_id', 'unknown')
        page_num = result['page_num']
        score = result['score']
        rank = result.get('rank', '?')
        print(f"  {rank}. 文档: {doc_id}, 第 {page_num} 页, 分数: {score:.4f}")


多文档检索 - Token级MaxSim聚合
查询数量: 1
检索模式: 跨文档跨页面


检索查询: 100%|██████████| 1/1 [00:00<00:00,  1.57it/s]

✓ 检索完成


跨文档跨页面检索结果

查询 1: 针对GRI403-3，各个公司有什么区别？
--------------------------------------------------------------------------------
[{'query': '针对GRI403-3，各个公司有什么区别？', 'doc_id': 'sanqi_esg', 'page_idx': 81, 'page_num': 82, 'page_id': 'sanqi_esg_page81', 'score': 12.601232528686523, 'rank': 1}, {'query': '针对GRI403-3，各个公司有什么区别？', 'doc_id': 'sanqi_esg', 'page_idx': 83, 'page_num': 84, 'page_id': 'sanqi_esg_page83', 'score': 11.565713882446289, 'rank': 2}, {'query': '针对GRI403-3，各个公司有什么区别？', 'doc_id': 'sanqi_esg', 'page_idx': 82, 'page_num': 83, 'page_id': 'sanqi_esg_page82', 'score': 10.914586067199707, 'rank': 3}, {'query': '针对GRI403-3，各个公司有什么区别？', 'doc_id': 'archi_esg', 'page_idx': 47, 'page_num': 48, 'page_id': 'archi_esg_page47', 'score': 9.616460800170898, 'rank': 4}, {'query': '针对GRI403-3，各个公司有什么区别？', 'doc_id': 'zhongxing_esg', 'page_idx': 141, 'page_num': 142, 'page_id': 'zhongxing_esg_page141', 'score': 9.4454345703125, 'rank': 5}, {'query': '针对GRI403-3，各个公司有什么区别？', 'doc_id': 'zhongx

### 7.2 每文档单页检索（保证文档多样性）

这种模式下，每个文档只返回得分最高的一页，保证结果来自不同文档

In [12]:
# 每文档单页检索
results_single = rag_pipeline.retrieve_multi_doc(
    queries=queries,
    top_k=5,
    single_page_per_doc=True,  # 每个文档只返回一页
)

# 显示结果
print("\n" + "="*80)
print("每文档单页检索结果（保证文档多样性）")
print("="*80)

for i, (query, query_results) in enumerate(zip(queries, results_single)):
    print(f"\n查询 {i+1}: {query}")
    print("-" * 80)
    
    for result in query_results:
        doc_id = result.get('doc_id', 'unknown')
        page_num = result['page_num']
        score = result['score']
        rank = result.get('rank', '?')
        print(f"  {rank}. 文档: {doc_id}, 第 {page_num} 页, 分数: {score:.4f}")


多文档检索 - Token级MaxSim聚合
查询数量: 1
检索模式: 每文档单页


Extracting query embeddings...:   0%|          | 0/1 [00:00<?, ?it/s]

检索查询: 100%|██████████| 1/1 [00:00<00:00,  1.55it/s]

✓ 检索完成


每文档单页检索结果（保证文档多样性）

查询 1: 针对GRI403-3，各个公司有什么区别？
--------------------------------------------------------------------------------
  1. 文档: sanqi_esg, 第 82 页, 分数: 9.6616
  2. 文档: zhongxing_esg, 第 142 页, 分数: 8.8490
  3. 文档: archi_esg, 第 48 页, 分数: 8.4259
  4. 文档: tencent_esg, 第 26 页, 分数: 3.1901


## 总结

本notebook演示了多文档检索的两种模式：

1. **跨文档跨页面检索** (`single_page_per_doc=False`)
   - ✅ 可以从同一文档返回多页
   - ✅ 适合答案分散在多页的场景
   - ✅ 真正的 multi-page 检索

2. **每文档单页检索** (`single_page_per_doc=True`)
   - ✅ 保证文档多样性
   - ✅ 适合答案在不同文档的场景
   - ✅ 每个文档只返回最相关的一页

参考 M3DocRAG 的设计理念，实现了灵活的多文档检索策略。